# 04 · Evaluate both agents with MLflow

One trace tells you about one answer. To decide whether an agent version is good, or better than another, you
need the same questions, the same checks, and a scorecard. `mlflow.genai.evaluate()` does that, given:
* a dataset;
* a `predict_fn` (here `OpenClawAgent.ask` or `HermesAgent.ask`);
* a list of scorers.

Each answer becomes a trace in an MLflow run, every scorer's verdict is an assessment on that trace, and the run
holds the aggregate metrics.

| scorer | kind | looks at |
|---|---|---|
| `no_stale_policy`, `no_disclaimer_opener`, `concise` | code | the answer |
| `read_right_policy`, `focused_reading` | code | the TOOL spans (which files it opened) |
| `has_key_facts` | code | the answer vs. the answer key (expectations) |
| `leads_with_answer` | Claude Opus 5.5 (Guidelines judge) | the answer |
| `correctness` | Claude Opus 5.5 (MLflow built-in judge) | the answer vs. the expected response |

Budget: 8 questions × 2 runtimes. That's about 3–5 minutes and roughly $1 in API usage.

In [1]:
from sia import notebook as nb

nb.setup(db="sdk.db", experiment="acme-support-sdk")

tracking_uri  sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db
experiment    acme-support-sdk (id 1)
backend       offline
agent_model   anthropic:/claude-haiku-5-5
judge_model   anthropic:/claude-opus-5-5
llm_judges    True


{'tracking_uri': 'sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db',
 'experiment': 'acme-support-sdk (id 1)',
 'backend': 'offline',
 'agent_model': 'anthropic:/claude-haiku-5-5',
 'judge_model': 'anthropic:/claude-opus-5-5',
 'llm_judges': True}

In [2]:
import mlflow
from sia import config
from sia.hermes_runtime import HermesAgent
from sia.openclaw_client import Gateway, OpenClawAgent

nb.ensure_production()
prompt = mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}@production")

gw = Gateway(nb.NB_STORE / "sdk" / "gateway", nb.sdk_workspace("openclaw")).start()
agents = {"openclaw": OpenClawAgent(gw), "hermes": HermesAgent(nb.sdk_workspace("hermes"))}
for agent in agents.values():
    agent.set_agents_md(prompt.template)
print(f"both agents run {config.PROMPT_NAME} v{prompt.version}")

both agents run acme-support-agents-md v1


## 1. An evaluation dataset

Eight customer questions from the scenario's `traffic` split, with:
* the reviewers' expected answer;
* their answer key (`key_facts`: patterns a correct answer must contain).

Stored as an MLflow **evaluation dataset**, so it's versioned and reusable, and every evaluation run links back to
it. The `topic` input tells `read_right_policy` which policy file the agent should have opened.

In [3]:
from sia.scenario import load_cases

EVAL_IDS = ['refund_window', 'prorated_refund', 'failed_payment', 'reset_expiry', 'lost_2fa', 'sso_pro', 'seats_pro', 'jira']
cases = [c for c in load_cases("traffic") if c.id in EVAL_IDS]
records = [{"inputs": {"question": c.question, "topic": c.topic},
            "expectations": {"expected_response": c.expected_response, "key_facts": c.must_include}}
           for c in cases]

from mlflow.genai.datasets import create_dataset, get_dataset

try:
    dataset = get_dataset(name="acme_support_sdk_eval")
except Exception:
    dataset = create_dataset(name="acme_support_sdk_eval",
                             experiment_id=[mlflow.get_experiment_by_name("acme-support-sdk").experiment_id])
dataset.merge_records(records)                 # idempotent: same inputs, same record
dataset.to_df()[["inputs", "expectations"]].head(8)

,inputs,expectations
0,{'question': 'What happens if my card payment ...,{'expected_response': 'It is retried 3 times o...
1,{'question': 'Does Acme Cloud integrate with J...,{'expected_response': 'No. Built-in integratio...
2,{'question': 'How long is a password reset lin...,"{'expected_response': '15 minutes.', 'key_fact..."
3,{'question': 'I bought Pro three weeks ago and...,{'expected_response': 'No. The refund window i...
4,{'question': 'How long can I wait to request a...,{'expected_response': 'Refunds are available w...
5,{'question': 'How many teammates can I invite ...,"{'expected_response': 'Up to 10 seats.', 'key_..."
6,{'question': 'Can we set up SSO with Okta on o...,{'expected_response': 'No. SSO is available on...
7,{'question': 'I lost my phone with my authenti...,{'expected_response': 'Use a backup code. With...


## 2. Scorers

The code scorers are ordinary Python functions decorated with MLflow's `@scorer`, in `src/sia/signals.py`. Here's
one that reads TOOL spans:

In [4]:
from sia import signals

print(nb.source(signals, "knowledge_reads"))
print(nb.source(signals, "read_right_policy"))

def knowledge_reads(trace) -> list[str]:
    reads = []
    for span in sorted(trace.data.spans, key=lambda sp: sp.start_time_ns):  # in the order they happened
        if span.span_type == SpanType.TOOL and isinstance(span.inputs, dict):
            path = str(span.inputs.get("path") or span.inputs.get("file_path") or "")
            if "knowledge/" in path:
                reads.append(path[path.index("knowledge/"):])
    return reads

@scorer
def read_right_policy(trace) -> Feedback:
    """Did the agent read the policy file for the question's topic before answering?"""
    topic = trace.info.tags.get("topic")
    if not topic:
        return Feedback(value=True, rationale="No topic tag on this trace.")
    expected, reads = f"knowledge/{topic}.md", knowledge_reads(trace)
    if expected in reads:
        return Feedback(value=True, rationale=f"Read {expected}.")
    if not reads:
        return Feedback(value=False, rationale=f"Answered without reading any policy file (expected {ex

In [5]:
from mlflow.genai.scorers import Correctness

scorers = [signals.no_stale_policy, signals.no_disclaimer_opener, signals.concise,
           signals.read_right_policy, signals.focused_reading, signals.has_key_facts]
if config.HAS_LLM:                              # Claude Opus 5.5 judges (MLFLOW_GENAI_JUDGE_DEFAULT_MODEL)
    scorers += [signals.leads_with_answer, Correctness()]
print(config.JUDGE_MODEL, [s.name for s in scorers])

anthropic:/claude-opus-5-5 ['no_stale_policy', 'no_disclaimer_opener', 'concise', 'read_right_policy', 'focused_reading', 'has_key_facts', 'leads_with_answer', 'correctness']


## 3. Evaluate

`predict_fn` receives the dataset's `inputs` as keyword arguments. MLflow wraps each call in a trace, and our
`acme_support_agent` span nests inside it, with tags for the topic, runtime, and prompt version. Each runtime gets
its own MLflow run.

The OpenClaw gateway handles several sessions at once, so its rows run in parallel. Hermes runs one at a time,
because it takes `AGENTS.md` from the process-wide working directory.

In [6]:
import os
import time


def predict_for(runtime):
    agent = agents[runtime]

    def predict_fn(question, topic):
        return agent.ask(question, tags={"topic": topic, "runtime": runtime, "source": "eval",
                                         "prompt_version": str(prompt.version)})
    return predict_fn


# MLflow normally runs the first row once as a "does predict_fn make a trace?" check; ask() always does.
os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "true"
results = {}
for runtime in agents:
    os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "4" if runtime == "openclaw" else "1"
    t = time.perf_counter()
    with mlflow.start_run(run_name=f"{runtime} · AGENTS.md v{prompt.version}"):
        mlflow.set_tags({"runtime": runtime, "prompt_version": str(prompt.version)})
        results[runtime] = mlflow.genai.evaluate(data=dataset, predict_fn=predict_for(runtime), scorers=scorers)
    print(f"{runtime}: {time.perf_counter() - t:.0f}s")

2026/10/20 07:30:44 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: openclaw · AGENTS.md v1
  Run ID: 049fbbd086964262806e97a9da552d46

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

openclaw: 23s



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: hermes · AGENTS.md v1
  Run ID: 519ec151c9264823b27e905fdd98ac59

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

hermes: 52s


## 4. The scorecard

The pass rate per scorer, per runtime. Same instructions, same questions, two runtimes. Where the columns
disagree, the runtime matters. Where both fail, the instructions are the problem, and that's what notebook 05
fixes.

In [7]:
import pandas as pd
from sia.gate import _passed


def pass_rates(result, scorers):
    df = result.result_df
    out = {}
    for s in scorers:
        vals = [v for v in map(_passed, df.get(f"{s.name}/value", [])) if v is not None]
        if vals:
            out[s.name] = sum(vals) / len(vals)
    return out


card = pd.DataFrame({rt: pass_rates(r, scorers) for rt, r in results.items()})
card.loc["OVERALL"] = card.mean()
card.style.format("{:.0%}")

,openclaw,hermes
no_stale_policy,50%,38%
no_disclaimer_opener,75%,62%
concise,50%,50%
read_right_policy,0%,0%
focused_reading,100%,100%
has_key_facts,0%,25%
leads_with_answer,25%,25%
correctness,0%,0%
OVERALL,38%,38%


In [8]:
rows = []
for runtime, r in results.items():
    for _, row in r.result_df.iterrows():
        fails = [s.name for s in scorers if _passed(row.get(f"{s.name}/value")) == 0.0]
        rows.append({"runtime": runtime, "question": row["request"]["question"][:55],
                     "failed": ", ".join(fails) or "-", "answer": str(row["response"])[:90]})
pd.DataFrame(rows)

,runtime,question,failed,answer
0,openclaw,What happens if my card payment fails?,"no_stale_policy, no_disclaimer_opener, concise...","Policies can vary by plan and region, so pleas..."
1,openclaw,Does Acme Cloud integrate with Jira?,"read_right_policy, has_key_facts, leads_with_a...",Thanks for reaching out! I don't have confirme...
2,openclaw,How long is a password reset link valid?,"read_right_policy, has_key_facts, correctness",Password reset links are valid for **24 hours*...
3,openclaw,I bought Pro three weeks ago and barely used i...,"no_stale_policy, concise, read_right_policy, h...",Good news: you bought Pro about three weeks ag...
4,openclaw,How long can I wait to request a refund?,"no_stale_policy, no_disclaimer_opener, read_ri...","Policies can vary by plan and region, so pleas..."
5,openclaw,How many teammates can I invite on Pro?,"read_right_policy, has_key_facts, correctness",I don't have the teammate limit for the Pro pl...
6,openclaw,Can we set up SSO with Okta on our Pro plan?,"concise, read_right_policy, has_key_facts, lea...","Thanks for reaching out! I'm sorry, I don't ha..."
7,openclaw,I lost my phone with my authenticator app. How...,"no_stale_policy, concise, read_right_policy, h...","I'm sorry, that's stressful. Policies can vary..."
8,hermes,What happens if my card payment fails?,"concise, read_right_policy, has_key_facts, lea...",I don't have a documented policy for failed ca...
9,hermes,Does Acme Cloud integrate with Jira?,"no_stale_policy, no_disclaimer_opener, concise...","Policies can vary by plan and region, so pleas..."


Every verdict has a rationale. These are the judges' and checks' explanations for one failing row. This is
the evidence the analyst in notebook 05 reads.

In [9]:
df = results["openclaw"].result_df
worst = df.loc[df[[f"{s.name}/value" for s in scorers if f"{s.name}/value" in df]].map(_passed).sum(axis=1).idxmin()]
print("Q:", worst["request"]["question"])
print("A:", worst["response"], "\n")
for s in scorers:
    if _passed(worst.get(f"{s.name}/value")) == 0.0:
        print(f"✗ {s.name}: {worst.get(f'{s.name}/rationale')}")

Q: What happens if my card payment fails?
A: Policies can vary by plan and region, so please double-check the Terms of Service for the exact rules that apply to your account.

I don't have specific information about what happens when a card payment fails, so I don't want to guess at your account's details. Here's what I'd suggest:

- **Check your billing page** for a failed-payment notice or an option to update your card.
- **Contact Acme Cloud billing support** for your account's grace period, retry schedule, and any effect on access.
- **Check with your bank or card issuer** if the decline might be due to a block, a limit, or an expired card.

If you tell me your plan and region, I can point you to the right support channel. I can also help with the other billing questions I know about, like refunds within 30 days of purchase. 

✗ no_stale_policy: Quotes outdated policy: refund window is 14 days since July 2026, not 30.
✗ no_disclaimer_opener: Opens with a disclaimer: 'Policies can v

## 5. Human labels on the same traces

Automated scorers are half the signal. The other half is people. The scenario's simulated support leads review
the OpenClaw answers. Each review is an MLflow assessment with a HUMAN source: a `quality` pass/fail with a
rationale, plus the correct answer. Notebook 05 reads judges and humans together.

In [10]:
from sia import reviewers
from sia.scenario import case_for_question

for trace_id, request, response in results["openclaw"].result_df[["trace_id", "request", "response"]].itertuples(index=False):
    v = reviewers.log_review(trace_id, case_for_question(request["question"]), str(response))
    print(f"{v.reviewer:>6} {'✓' if v.passed else '✗'}  {request['question'][:60]}")

 maria ✗  What happens if my card payment fails?
   dev ✗  Does Acme Cloud integrate with Jira?


 maria ✗  How long is a password reset link valid?
   dev ✗  I bought Pro three weeks ago and barely used it. Can I get a
   dev ✗  How long can I wait to request a refund?
 maria ✗  How many teammates can I invite on Pro?
   sam ✗  Can we set up SSO with Okta on our Pro plan?
 maria ✗  I lost my phone with my authenticator app. How do I get back


In [11]:
agents["openclaw"].close()

**Next:** 05 · Improve and gate. Turn these failures into a better `AGENTS.md`, and prove it on data the
analyst never saw.